In [66]:
import pandas as pd 
import numpy as np

In [67]:
df = pd.read_csv("dataset.csv")
fraud_probability = df["fraud_probability"]
print("Laundering typology tells us what kind of fraud was performed.")
(((df["is_fraud"] == 0) == (df["laundering_typology"] == "normal")) & ((df["is_fraud"] == 1) == (df["laundering_typology"] != "normal"))).all()

Laundering typology tells us what kind of fraud was performed.


np.True_

In [68]:
df = df.drop(columns = ["fraud_probability", "laundering_typology"])
df.columns

Index(['type', 'category', 'orig', 'dest', 'old_balance', 'new_balance',
       'is_fraud', 'payment_method', 'location.city', 'location.state',
       'location.postcode', 'device_info.type', 'device_info.os',
       'device_info.ip_address', 'risk_indicators.amount_vs_average',
       'risk_indicators.customer_risk_score', 'risk_indicators.category_risk',
       'risk_indicators.risk_score', 'risk_indicators.unusual_time',
       'risk_indicators.unusual_location', 'merchant_info.merchant_id',
       'merchant_info.category', 'merchant_info.risk_level',
       'merchant_info.avg_transaction', 'datetime'],
      dtype='object')

1. Location preprocessing. Every city belongs to a unique state. Therefore we combine city and state and create a single categorical column. Use relative frequencies for postcodes.

In [69]:
df["city_state"] = (df["location.city"] + "_" + df["location.state"]).astype("category").cat.codes
df["postcode"] = df["location.postcode"]
df = df.drop(columns = ["location.city", "location.state", "location.postcode"])
print(f"Number of unique cities: {df["city_state"].nunique()}")

Number of unique cities: 6


2. Device type preprocessing. We combine type and os and create a single categorical column. Use subnet (for example 192.168.1.5 -> 192.168) for ip addresses, and then convert them to categories.

In [70]:
df["device"] = (df["device_info.type"] + "_" + df["device_info.os"]).astype("category").cat.codes
print(f"Number of unique devices: {df["device"].nunique()}")
df["ip_subnet"] = df["device_info.ip_address"].astype(str).str.extract(r"^(\d+\.\d+)")[0]
df["ip_subnet"] = df["ip_subnet"].astype("category").cat.codes
print(f"Number of unique ip addresses: {df["device_info.ip_address"].nunique()}")
print(f"Number of unique ip subnets: {df["ip_subnet"].nunique()}")
subnets_per_orig = df.groupby("orig")["ip_subnet"].nunique()
print(f"Average subnets per user: {subnets_per_orig.mean()}")
print(f"Users with only 1 subnet: {(subnets_per_orig == 1).sum()}")

Number of unique devices: 12
Number of unique ip addresses: 1090043
Number of unique ip subnets: 64516
Average subnets per user: 108.9081
Users with only 1 subnet: 0


In [71]:
df = df.drop(columns = ["device_info.type", "device_info.os", "device_info.ip_address"])

3. Convert type, category, and payment_method into categorical data.

In [72]:
df["type"] = df["type"].astype("category").cat.codes
df["category"] = df["category"].astype("category").cat.codes
df["payment_method"] = df["payment_method"].astype("category").cat.codes
print(f"Number of unique types: {df["type"].nunique()}")
print(f"Number of unique categories: {df["category"].nunique()}")
print(f"Number of unique payment methods: {df["payment_method"].nunique()}")

Number of unique types: 8
Number of unique categories: 11
Number of unique payment methods: 3


4. Analysis of risk_indicator columns, as they are the outputs of the AMLNet models.

In [73]:
risk_columns = [c for c in df.columns if c.startswith("risk_indicators.")]
temp_df = df[risk_columns + ["is_fraud"]].copy()

temp_df["risk_indicators.category_risk"] = temp_df["risk_indicators.category_risk"].astype("category").cat.codes
temp_df["risk_indicators.unusual_time"] = temp_df["risk_indicators.unusual_time"].astype("category").cat.codes
temp_df["risk_indicators.unusual_location"] = temp_df["risk_indicators.unusual_location"].astype("category").cat.codes

leakage_analysis = temp_df.corr()["is_fraud"].drop("is_fraud").sort_values(ascending=False)
leakage_analysis

risk_indicators.amount_vs_average      0.106359
risk_indicators.risk_score             0.078350
risk_indicators.category_risk          0.045435
risk_indicators.customer_risk_score   -0.047015
risk_indicators.unusual_time                NaN
risk_indicators.unusual_location            NaN
Name: is_fraud, dtype: float64

Risk indicators are still the results of the AMLNet models, and therefore they should be removed from our baseline model.

In [74]:
df = df.drop(columns = risk_columns)
df.head()

,type,category,orig,dest,old_balance,new_balance,is_fraud,payment_method,merchant_info.merchant_id,merchant_info.category,merchant_info.risk_level,merchant_info.avg_transaction,datetime,city_state,postcode,device,ip_subnet
0,2,5,C8083,C7053,455489.321571,455190.479531,0,0,NaN,NaN,NaN,NaN,2025-02-04 12:22:36,5,2755,6,3021
1,2,7,C5575,C1117,229508.291214,229415.203298,0,1,NaN,NaN,NaN,NaN,2025-02-04 12:44:31,1,4976,8,30776
2,3,3,C1549,C1423,202568.806856,202413.161992,0,2,NaN,NaN,NaN,NaN,2025-02-04 12:48:01,5,2927,6,23741
3,0,2,C7435,C6390,491560.600203,491260.841131,0,1,NaN,NaN,NaN,NaN,2025-02-04 12:57:48,5,2332,5,43287
4,2,5,C8083,C5946,455190.479531,455016.763916,0,1,NaN,NaN,NaN,NaN,2025-02-04 12:58:37,5,2572,10,4091


In [75]:
df_orig = df.copy()
df_orig["account"] = df_orig["orig"]
df_orig["is_dest"] = 0

df_dest = df.copy()
df_dest["account"] = df_dest["dest"]
df_dest["is_dest"] = 1

df_nodes = pd.concat([
    df_orig.drop(columns=["orig", "dest"]), 
    df_dest.drop(columns=["orig", "dest"])
], ignore_index=True)

df_nodes.head()

,type,category,old_balance,new_balance,is_fraud,payment_method,merchant_info.merchant_id,merchant_info.category,merchant_info.risk_level,merchant_info.avg_transaction,datetime,city_state,postcode,device,ip_subnet,account,is_dest
0,2,5,455489.321571,455190.479531,0,0,NaN,NaN,NaN,NaN,2025-02-04 12:22:36,5,2755,6,3021,C8083,0
1,2,7,229508.291214,229415.203298,0,1,NaN,NaN,NaN,NaN,2025-02-04 12:44:31,1,4976,8,30776,C5575,0
2,3,3,202568.806856,202413.161992,0,2,NaN,NaN,NaN,NaN,2025-02-04 12:48:01,5,2927,6,23741,C1549,0
3,0,2,491560.600203,491260.841131,0,1,NaN,NaN,NaN,NaN,2025-02-04 12:57:48,5,2332,5,43287,C7435,0
4,2,5,455190.479531,455016.763916,0,1,NaN,NaN,NaN,NaN,2025-02-04 12:58:37,5,2572,10,4091,C8083,0


In [76]:
account_counts = df_nodes.groupby("account")["is_fraud"].value_counts().unstack(fill_value=0)
viable_accounts = account_counts[(account_counts[1] >= 3) & (account_counts[0] >= 5)]
print(f"Number of unique accounts: {len(account_counts)}")
print(f"Number of unique accounts involved in at least 3 fraudulent and 5 non-fraudulent transactions: {len(viable_accounts)}")
viable_accounts.head()

Number of unique accounts: 11000
Number of unique accounts involved in at least 3 fraudulent and 5 non-fraudulent transactions: 292


is_fraud,0,1
account,,
C1076,127,11
C1089,229,11
C109,309,4
C1123,248,8
C1130,175,4


In [77]:
user_0 = df_nodes[df_nodes["account"] == viable_accounts.index[0]].copy()
user_0 = user_0.drop(columns = ["account"]).reset_index(drop = True)
print(f"Number of transactions: {len(user_0)}")
print(f"Number of unique cities: {user_0["city_state"].nunique()}")
print(f"Number of unique postcodes: {user_0["postcode"].nunique()}")
print(f"Number of unique devices: {user_0["device"].nunique()}")
print(f"Number of unique ip subnets: {user_0["ip_subnet"].nunique()}")
print(f"Number of times as origin: {(user_0["is_dest"] == 0).sum()}")
print(f"Number of times as destination: {(user_0["is_dest"] == 1).sum()}")
print(f"Number of frauds committed as origin: {((user_0["is_dest"] == 0) & (user_0["is_fraud"] == 1)).sum()}")
print(f"Number of frauds committed as destination: {((user_0["is_dest"] == 1) & (user_0["is_fraud"] == 1)).sum()}")
user_0.head()

Number of transactions: 138
Number of unique cities: 1
Number of unique postcodes: 130
Number of unique devices: 12
Number of unique ip subnets: 138
Number of times as origin: 70
Number of times as destination: 68
Number of frauds committed as origin: 11
Number of frauds committed as destination: 0


,type,category,old_balance,new_balance,is_fraud,payment_method,merchant_info.merchant_id,merchant_info.category,merchant_info.risk_level,merchant_info.avg_transaction,datetime,city_state,postcode,device,ip_subnet,is_dest
0,2,5,480374.255176,480175.012937,0,2,NaN,NaN,NaN,NaN,2025-02-05 01:57:40,5,2569,8,36133,0
1,5,1,480175.012937,479562.985697,0,1,M276,Education,low,500.0,2025-02-15 16:53:28,5,2849,5,53315,0
2,2,5,479562.985697,479497.806775,0,2,NaN,NaN,NaN,NaN,2025-02-14 00:59:58,5,2452,10,23576,0
3,1,5,479497.806775,479344.233527,0,2,NaN,NaN,NaN,NaN,2025-02-21 22:35:02,5,2159,11,6703,0
4,7,7,479189.424637,479112.720338,0,0,NaN,NaN,NaN,NaN,2025-02-22 13:06:33,5,2135,4,55505,0


In [78]:
user_1 = df_nodes[df_nodes["account"] == viable_accounts.index[1]].copy()
user_1 = user_1.drop(columns = ["account"]).reset_index(drop = True)
print(f"Number of transactions: {len(user_1)}")
print(f"Number of unique cities: {user_1["city_state"].nunique()}")
print(f"Number of unique postcodes: {user_1["postcode"].nunique()}")
print(f"Number of unique devices: {user_1["device"].nunique()}")
print(f"Number of unique ip subnets: {user_1["ip_subnet"].nunique()}")
print(f"Number of times as origin: {(user_1["is_dest"] == 0).sum()}")
print(f"Number of times as destination: {(user_1["is_dest"] == 1).sum()}")
print(f"Number of frauds committed as origin: {((user_1["is_dest"] == 0) & (user_1["is_fraud"] == 1)).sum()}")
print(f"Number of frauds committed as destination: {((user_1["is_dest"] == 1) & (user_1["is_fraud"] == 1)).sum()}")
user_1.head()

Number of transactions: 240
Number of unique cities: 1
Number of unique postcodes: 210
Number of unique devices: 12
Number of unique ip subnets: 239
Number of times as origin: 153
Number of times as destination: 87
Number of frauds committed as origin: 11
Number of frauds committed as destination: 0


,type,category,old_balance,new_balance,is_fraud,payment_method,merchant_info.merchant_id,merchant_info.category,merchant_info.risk_level,merchant_info.avg_transaction,datetime,city_state,postcode,device,ip_subnet,is_dest
0,2,4,474855.471914,473264.639497,0,0,M26,Housing,low,2000.0,2025-02-04 23:24:01,2,3067,5,14293,0
1,7,4,473055.067940,470872.824406,0,1,M487,Housing,low,2000.0,2025-02-05 00:05:03,2,3887,11,42021,0
2,2,5,473136.330134,473055.067940,0,1,NaN,NaN,NaN,NaN,2025-02-05 00:07:40,2,3067,4,25467,0
3,2,5,473264.639497,473136.330134,0,1,NaN,NaN,NaN,NaN,2025-02-05 01:37:45,2,3374,6,55091,0
4,2,5,469911.294849,469443.663256,0,2,NaN,NaN,NaN,NaN,2025-02-07 10:03:22,2,3375,6,18151,0


In [79]:
user_2 = df_nodes[df_nodes["account"] == viable_accounts.index[2]].copy()
user_2 = user_2.drop(columns = ["account"]).reset_index(drop = True)
print(f"Number of transactions: {len(user_2)}")
print(f"Number of unique cities: {user_2["city_state"].nunique()}")
print(f"Number of unique postcodes: {user_2["postcode"].nunique()}")
print(f"Number of unique devices: {user_2["device"].nunique()}")
print(f"Number of unique ip subnets: {user_2["ip_subnet"].nunique()}")
print(f"Number of times as origin: {(user_2["is_dest"] == 0).sum()}")
print(f"Number of times as destination: {(user_2["is_dest"] == 1).sum()}")
print(f"Number of frauds committed as origin: {((user_2["is_dest"] == 0) & (user_2["is_fraud"] == 1)).sum()}")
print(f"Number of frauds committed as destination: {((user_2["is_dest"] == 1) & (user_2["is_fraud"] == 1)).sum()}")
user_2.head()

Number of transactions: 313
Number of unique cities: 1
Number of unique postcodes: 267
Number of unique devices: 12
Number of unique ip subnets: 311
Number of times as origin: 220
Number of times as destination: 93
Number of frauds committed as origin: 1
Number of frauds committed as destination: 3


,type,category,old_balance,new_balance,is_fraud,payment_method,merchant_info.merchant_id,merchant_info.category,merchant_info.risk_level,merchant_info.avg_transaction,datetime,city_state,postcode,device,ip_subnet,is_dest
0,2,7,438076.606789,437956.585798,0,1,NaN,NaN,NaN,NaN,2025-02-06 14:37:56,5,2166,7,25355,0
1,2,4,436641.193772,434731.882537,0,2,M695,Housing,low,2000.0,2025-02-06 16:25:26,5,2866,7,52042,0
2,2,4,434731.882537,432865.804986,0,2,M600,Housing,low,2000.0,2025-02-06 17:23:58,5,2787,7,9750,0
3,2,4,437956.585798,436641.193772,0,0,M426,Housing,low,2000.0,2025-02-07 00:01:18,5,2032,11,37351,0
4,2,5,432865.804986,432776.025215,0,0,NaN,NaN,NaN,NaN,2025-02-07 04:20:43,5,2068,11,10577,0
